# Sequential Modelling with Hidden Markov Models(HMM)

## Motivation

Previous experiments were conducted using memoryless classifiers 
(e.g. SVM / Random Forest), where each frame was treated independently. And the robot was acting boring.

However, robot facial and head states evolve sequentially over time. To address this limitation, We address it using HMM.

The observable variables correspond to the extracted facial and head features.
The hidden states are assumed to represent latent robot expression states.

## Feature Set

We focus on:

- Facial features
- Head pose features

Intensity is excluded at this stage as it already performs satisfactorily.

All features are scaled using StandardScaler prior to training.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from hmmlearn import hmm

import warnings
warnings.filterwarnings('ignore')


FEATURES_PATH = '/Users/miaaa/Desktop/music robot/furhat_music_robot/features_v3.csv'
METADATA_PATH = '/Users/miaaa/Desktop/music robot/furhat_music_robot/annotations/metadata.csv'

# ─── Load ───
features = pd.read_csv(FEATURES_PATH)
metadata = pd.read_csv(METADATA_PATH)
features['head_movement'] = features['head_movement'].str.strip()
features['split'] = features['song_name'].map(dict(zip(metadata['title'], metadata['split'])))

# Merge
merge_map = {
    'shake, nod': 'nod',
    'sway, nod':  'sway',
    'look down':  'none',
    'look up':    'none',
}
features['head_movement'] = features['head_movement'].replace(merge_map)
CLASSES = sorted(features['head_movement'].unique())
print('Classes:', CLASSES)

train_data = features[features['split'] == 'Train']
val_data   = features[features['split'] == 'Validation']
test_data  = features[features['split'] == 'Test']

print(f'\nTrain: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}')
print('\nTraining distribution:')
vc = train_data['head_movement'].value_counts()
for label, cnt in vc.items():
    print(f'  {label:10s}: {cnt:3d}  ({cnt/len(train_data)*100:.1f}%)')

# all features
LABEL_COLS = ['song_name', 'start_time', 'end_time', 'duration',
              'head_movement', 'facial_expression', 'intensity', 'split']

TOP_FEATURES = [c for c in features.columns if c not in LABEL_COLS]

print(f'Using {len(TOP_FEATURES)} features')

X_train_raw = train_data[TOP_FEATURES].values
X_val_raw   = val_data[TOP_FEATURES].values
X_test_raw  = test_data[TOP_FEATURES].values
y_train = train_data['head_movement'].values
y_val   = val_data['head_movement'].values
y_test  = test_data['head_movement'].values

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val   = scaler.transform(X_val_raw)
X_test  = scaler.transform(X_test_raw)

# Baseline — DummyClassifier
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)
dummy_val_acc  = accuracy_score(y_val,  dummy.predict(X_val))
dummy_test_acc = accuracy_score(y_test, dummy.predict(X_test))
print(f'\nDummy  — Val: {dummy_val_acc:.4f}  Test: {dummy_test_acc:.4f}')

# Model 1 — Random Forest (improved)
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

rf_train_acc = accuracy_score(y_train, rf.predict(X_train))
rf_val_acc   = accuracy_score(y_val,   rf.predict(X_val))
rf_test_acc  = accuracy_score(y_test,  rf.predict(X_test))
print(f'\nRF     — Train: {rf_train_acc:.4f}  Val: {rf_val_acc:.4f}  Test: {rf_test_acc:.4f}')
print('\nRF Validation Classification Report:')
print(classification_report(y_val, rf.predict(X_val)))

# Model 2 — HMM (per-class Gaussian)
def prepare_sequences(df, feature_cols, fitted_scaler):
    seqs, seq_labels = [], []
    for _, group in df.groupby('song_name'):
        group = group.sort_values('start_time')
        X = fitted_scaler.transform(group[feature_cols].values)
        seqs.append(X)
        seq_labels.append(group['head_movement'].values)
    return seqs, seq_labels

train_seqs, train_seq_labels = prepare_sequences(train_data, TOP_FEATURES, scaler)
val_seqs,   val_seq_labels   = prepare_sequences(val_data,   TOP_FEATURES, scaler)
test_seqs,  test_seq_labels  = prepare_sequences(test_data,  TOP_FEATURES, scaler)

# Train one Gaussian HMM per class
print('\nTraining HMMs...')
class_hmms = {}
for cls in CLASSES:
    cls_seqs, cls_lengths = [], []
    for seq, labels in zip(train_seqs, train_seq_labels):
        mask = (labels == cls)
        if mask.sum() > 0:
            cls_seqs.append(seq[mask])
            cls_lengths.append(int(mask.sum()))

    if not cls_seqs:
        print(f'  WARNING: no samples for "{cls}", skipping')
        continue

    X_cls  = np.vstack(cls_seqs)
    n_comp = min(3, len(X_cls))
    model  = hmm.GaussianHMM(n_components=n_comp, covariance_type='diag',
                              n_iter=100, random_state=42, verbose=False)
    model.fit(X_cls, cls_lengths)
    class_hmms[cls] = model
    print(f'  "{cls:10s}" — {X_cls.shape[0]:3d} segments, {n_comp} hidden states')

def hmm_predict(seqs, seq_labels, class_hmms):
    y_true, y_pred = [], []
    for seq, labels in zip(seqs, seq_labels):
        for segment, true_label in zip(seq, labels):
            x = segment.reshape(1, -1)
            best_cls, best_score = None, -np.inf
            for cls, model in class_hmms.items():
                try:
                    score = model.score(x)
                except Exception:
                    score = -np.inf
                if score > best_score:
                    best_score, best_cls = score, cls
            y_true.append(true_label)
            y_pred.append(best_cls)
    return np.array(y_true), np.array(y_pred)

y_val_true,  y_val_hmm  = hmm_predict(val_seqs,  val_seq_labels,  class_hmms)
y_test_true, y_test_hmm = hmm_predict(test_seqs, test_seq_labels, class_hmms)

hmm_val_acc  = accuracy_score(y_val_true,  y_val_hmm)
hmm_test_acc = accuracy_score(y_test_true, y_test_hmm)
print(f'\nHMM    — Val: {hmm_val_acc:.4f}  Test: {hmm_test_acc:.4f}')
print('\nHMM Validation Classification Report:')
print(classification_report(y_val_true, y_val_hmm))

# ════════════════════════════════════
# Summary
# ════════════════════════════════════
print('\n')
print(f'{"Model":<25} {"Val":>6}  {"Test":>6}')
print(f'{"Dummy (baseline)":<25} {dummy_val_acc:>6.4f}  {dummy_test_acc:>6.4f}')
print(f'{"Random Forest (improved)":<25} {rf_val_acc:>6.4f}  {rf_test_acc:>6.4f}')
print(f'{"HMM":<25} {hmm_val_acc:>6.4f}  {hmm_test_acc:>6.4f}')


Classes: ['nod', 'none', 'shake', 'sway']

Train: 314 | Val: 64 | Test: 52

Training distribution:
  none      : 107  (34.1%)
  nod       : 103  (32.8%)
  sway      :  62  (19.7%)
  shake     :  42  (13.4%)
Using 111 features

Dummy  — Val: 0.3125  Test: 0.3077

RF     — Train: 1.0000  Val: 0.5156  Test: 0.5385

RF Validation Classification Report:
              precision    recall  f1-score   support

         nod       0.52      0.61      0.56        23
        none       0.64      0.80      0.71        20
       shake       0.12      0.12      0.12         8
        sway       0.50      0.15      0.24        13

    accuracy                           0.52        64
   macro avg       0.45      0.42      0.41        64
weighted avg       0.50      0.52      0.49        64


Training HMMs...


Model is not converging.  Current: -15476.389708473971 is not greater than -15476.38969026561. Delta is -1.8208360415883362e-05


  "nod       " — 103 segments, 3 hidden states
  "none      " — 107 segments, 3 hidden states
  "shake     " —  42 segments, 3 hidden states
  "sway      " —  62 segments, 3 hidden states

HMM    — Val: 0.4531  Test: 0.4231

HMM Validation Classification Report:
              precision    recall  f1-score   support

         nod       0.60      0.26      0.36        23
        none       0.59      0.65      0.62        20
       shake       0.25      0.25      0.25         8
        sway       0.33      0.62      0.43        13

    accuracy                           0.45        64
   macro avg       0.44      0.44      0.42        64
weighted avg       0.50      0.45      0.44        64



Model                        Val    Test
Dummy (baseline)          0.3125  0.3077
Random Forest (improved)  0.5156  0.5385
HMM                       0.4531  0.4231


# Head Movement Classifier — Results

## Data
Train 314 | Val 64 | Test 52 | Classes: `nod`, `none`, `shake`, `sway`

## Model Comparison

| Model | Val | Test |
|-------|-----|------|
| Dummy (baseline) | 0.31 | 0.31 |
| HMM per-segment | 0.45 | 0.42 |
| **Random Forest** | **0.52** | **0.54** |

## Per-Class Recall (Validation)

| Class | RF | HMM |
|-------|----|-----|
| nod   | **0.61** | 0.26 |
| none  | **0.80** | 0.65 |
| shake | 0.12 | 0.25 |
| sway  | 0.15 | **0.62** |

RF wins overall; HMM only beats RF on `sway`.

## HMM Sequential Decoding — Why It Failed

Window sweep result: **w=1 is best** (Val 0.4531). Larger windows only make it worse.

The problem: each HMM was trained on segments of a single class only. During inference, a window of size >1 always contains segments from mixed classes — the model was never trained on sequences like that, so it gets confused.

## What To Try Next

| Approach | Note |
|----------|------|
| **RF + temporal smoothing** | Take RF predictions, apply majority vote over last N segments. No retraining needed. |
| **RF → Viterbi hybrid** | Feed RF class probabilities into a Viterbi decoder. Best of both worlds. |
| **CRF** | Train directly on full song sequences with label transitions. Correct model for this task. |

Most practical next step: **RF + temporal smoothing** (1 extra cell, no retraining).

In [2]:

# ════════════════════════════════════════════════════════════════════
# Model 3 — HMM with TRUE Sequential Decoding (Viterbi over full song)
# ════════════════════════════════════════════════════════════════════
#
# Previous HMM scored each segment independently (reshape(1,-1)).
# Here we feed the ENTIRE song sequence to each class HMM and use
# Viterbi to decode the most likely class label at every time step.
#
# Approach: sliding-window majority vote over a window of W segments.
#   For each window [t-W+1 .. t], pick the class whose HMM gives the
#   highest log-likelihood for that subsequence. This lets transition
#   probabilities and temporal smoothing do real work.
# ════════════════════════════════════════════════════════════════════

WINDOW = 1   

def hmm_predict_sequential(seqs, seq_labels, class_hmms, window=WINDOW):
    """
    For each song sequence, slide a window of `window` consecutive segments.
    Score each window against every class HMM; assign the class with highest
    log-likelihood to the *last* segment in the window.
    """
    y_true, y_pred = [], []

    for seq, labels in zip(seqs, seq_labels):
        T = len(seq)
        for t in range(T):
            t_start = max(0, t - window + 1)
            window_obs = seq[t_start : t + 1]   # shape (w, n_features)

            best_cls, best_score = None, -np.inf
            for cls, model in class_hmms.items():
                try:
                    score = model.score(window_obs)
                except Exception:
                    score = -np.inf
                if score > best_score:
                    best_score, best_cls = score, cls

            y_true.append(labels[t])
            y_pred.append(best_cls)

    return np.array(y_true), np.array(y_pred)


print(f'HMM Sequential Decoding (window={WINDOW})')
print('─' * 45)

y_val_true_seq,  y_val_seq  = hmm_predict_sequential(val_seqs,  val_seq_labels,  class_hmms)
y_test_true_seq, y_test_seq = hmm_predict_sequential(test_seqs, test_seq_labels, class_hmms)

hmm_seq_val_acc  = accuracy_score(y_val_true_seq,  y_val_seq)
hmm_seq_test_acc = accuracy_score(y_test_true_seq, y_test_seq)

print(f'HMM-Seq — Val: {hmm_seq_val_acc:.4f}  Test: {hmm_seq_test_acc:.4f}')
print('\nValidation Classification Report:')
print(classification_report(y_val_true_seq, y_val_seq))

# ── sweep windows to find best ──────────────────────────────────────
print('Window sweep (Val accuracy):')
best_w, best_acc = 1, 0
for w in range(1, 8):
    yt, yp = hmm_predict_sequential(val_seqs, val_seq_labels, class_hmms, window=w)
    acc = accuracy_score(yt, yp)
    marker = ' ◄' if w == WINDOW else ''
    print(f'  w={w}: {acc:.4f}{marker}')
    if acc > best_acc:
        best_acc, best_w = acc, w

print(f'\nBest window on Val: w={best_w}  acc={best_acc:.4f}')
yt_best, yp_best = hmm_predict_sequential(test_seqs, test_seq_labels, class_hmms, window=best_w)
print(f'  → Test acc with w={best_w}: {accuracy_score(yt_best, yp_best):.4f}')

# ── updated summary ─────────────────────────────────────────────────
print('\n')
print(f'{"Model":<30} {"Val":>6}  {"Test":>6}')
print('-' * 50)
print(f'{"Dummy (baseline)":<30} {dummy_val_acc:>6.4f}  {dummy_test_acc:>6.4f}')
print(f'{"Random Forest":<30} {rf_val_acc:>6.4f}  {rf_test_acc:>6.4f}')
print(f'{"HMM (per-segment)":<30} {hmm_val_acc:>6.4f}  {hmm_test_acc:>6.4f}')
print(f'{"HMM-Seq (window={:d})":<30} {hmm_seq_val_acc:>6.4f}  {hmm_seq_test_acc:>6.4f}'.format(WINDOW))



HMM Sequential Decoding (window=1)
─────────────────────────────────────────────
HMM-Seq — Val: 0.4531  Test: 0.4231

Validation Classification Report:
              precision    recall  f1-score   support

         nod       0.60      0.26      0.36        23
        none       0.59      0.65      0.62        20
       shake       0.25      0.25      0.25         8
        sway       0.33      0.62      0.43        13

    accuracy                           0.45        64
   macro avg       0.44      0.44      0.42        64
weighted avg       0.50      0.45      0.44        64

Window sweep (Val accuracy):
  w=1: 0.4531 ◄
  w=2: 0.3438
  w=3: 0.3438
  w=4: 0.3438
  w=5: 0.3125
  w=6: 0.2969
  w=7: 0.3281

Best window on Val: w=1  acc=0.4531
  → Test acc with w=1: 0.4231


Model                             Val    Test
--------------------------------------------------
Dummy (baseline)               0.3125  0.3077
Random Forest                  0.5156  0.5385
HMM (per-segment)         

In [3]:

# ════════════════════════════════════════════════════════════════════
# Facial Expression — HMM (per-class Gaussian) + feature set search
# ════════════════════════════════════════════════════════════════════

# ── helper: same as merge logic in RF script ────────────────────────
def merge_facial(expr):
    e = str(expr).strip().lower()
    if 'angry' in e or 'disgust' in e:        return 'negative'
    if 'big smile' in e or ('browraise' in e and 'smile' in e): return 'big_smile'
    if 'browraise' in e or 'surprise' in e or 'oh' in e:        return 'surprise'
    if 'frown' in e or 'sad' in e:            return 'frown'
    if 'thoughtful' in e or 'confused' in e:  return 'thoughtful'
    if 'smile' in e:                          return 'smile'
    return 'neutral'

# reload raw data so this cell is self-contained
f2 = pd.read_csv(FEATURES_PATH)
m2 = pd.read_csv(METADATA_PATH)
f2['facial_expression'] = f2['facial_expression'].str.strip()
f2['split'] = f2['song_name'].map(dict(zip(m2['title'], m2['split'])))
f2['facial_merged'] = f2['facial_expression'].apply(merge_facial)

# ── merge rare classes (negative=6, thoughtful=22 too small for HMM) ─
facial_merge = {
    'negative':   'frown',      # both negative valence
    'thoughtful': 'neutral',    # both low-arousal neutral
    'surprise':   'big_smile',  # both high-arousal positive
}
f2['facial_label'] = f2['facial_merged'].replace(facial_merge)
FACIAL_CLASSES = sorted(f2['facial_label'].unique())
print('Facial classes after merge:', FACIAL_CLASSES)

ftr = f2[f2['split'] == 'Train']
fva = f2[f2['split'] == 'Validation']
fte = f2[f2['split'] == 'Test']

print('\nTraining distribution:')
vc = ftr['facial_label'].value_counts()
for lbl, cnt in vc.items():
    print(f'  {lbl:12s}: {cnt:3d}  ({cnt/len(ftr)*100:.1f}%)')

# ── feature groups to try ────────────────────────────────────────────
LABEL_COLS2 = ['song_name', 'start_time', 'end_time', 'duration',
               'head_movement', 'facial_expression', 'intensity',
               'facial_merged', 'facial_label', 'split']
ALL_FEATS = [c for c in f2.columns if c not in LABEL_COLS2]

mfcc_feats    = [c for c in ALL_FEATS if c.startswith('mfcc') and 'delta' not in c]
delta_feats   = [c for c in ALL_FEATS if 'delta' in c]
chroma_feats  = [c for c in ALL_FEATS if c.startswith('chroma')]
spectral_feats= [c for c in ALL_FEATS if any(c.startswith(p) for p in
                  ['rms','spectral','zcr','tempo'])]

FEAT_SETS = {
    'all_111':            ALL_FEATS,
    'mfcc_only':          mfcc_feats,
    'mfcc+chroma':        mfcc_feats + chroma_feats,
    'mfcc+delta':         mfcc_feats + delta_feats,
    'mfcc+spectral':      mfcc_feats + spectral_feats,
    'no_delta2':          [c for c in ALL_FEATS if 'delta2' not in c],
}

# ── train + evaluate one HMM config ─────────────────────────────────
def run_facial_hmm(feat_cols, n_comp=3, label=''):
    sc = StandardScaler()
    Xtr = sc.fit_transform(ftr[feat_cols].values)
    Xva = sc.transform(fva[feat_cols].values)
    Xte = sc.transform(fte[feat_cols].values)
    ytr = ftr['facial_label'].values
    yva = fva['facial_label'].values
    yte = fte['facial_label'].values

    # train per-class HMMs
    models = {}
    for cls in FACIAL_CLASSES:
        segs = Xtr[ytr == cls]
        if len(segs) < n_comp:
            continue
        k = min(n_comp, len(segs))
        m = hmm.GaussianHMM(n_components=k, covariance_type='diag',
                             n_iter=200, random_state=42, verbose=False)
        m.fit(segs, [len(segs)])
        models[cls] = m

    # predict (per-segment)
    def predict(X):
        preds = []
        for x in X:
            obs = x.reshape(1, -1)
            best, best_s = None, -np.inf
            for cls, m in models.items():
                try: s = m.score(obs)
                except: s = -np.inf
                if s > best_s:
                    best_s, best = s, cls
            preds.append(best)
        return np.array(preds)

    val_acc  = accuracy_score(yva, predict(Xva))
    test_acc = accuracy_score(yte, predict(Xte))
    return val_acc, test_acc, models, sc, yva, predict(Xva)

# ── dummy baseline ───────────────────────────────────────────────────
from sklearn.dummy import DummyClassifier as DC
_sc = StandardScaler()
_Xtr = _sc.fit_transform(ftr[ALL_FEATS].values)
_Xva = _sc.transform(fva[ALL_FEATS].values)
_Xte = _sc.transform(fte[ALL_FEATS].values)
_ytr, _yva, _yte = ftr['facial_label'].values, fva['facial_label'].values, fte['facial_label'].values
_d = DC(strategy='most_frequent', random_state=42).fit(_Xtr, _ytr)
print(f'\nDummy — Val: {accuracy_score(_yva,_d.predict(_Xva)):.4f}  Test: {accuracy_score(_yte,_d.predict(_Xte)):.4f}')

# ── RF baseline (same merged labels) ────────────────────────────────
_rf = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_split=5,
                              min_samples_leaf=2, max_features='sqrt',
                              random_state=42, n_jobs=-1).fit(_Xtr, _ytr)
rf_fval  = accuracy_score(_yva, _rf.predict(_Xva))
rf_ftest = accuracy_score(_yte, _rf.predict(_Xte))
print(f'RF    — Val: {rf_fval:.4f}  Test: {rf_ftest:.4f}')

# ── sweep feature sets ───────────────────────────────────────────────
print('\nFeature set sweep (n_components=3):')
print(f'  {"Feature set":<20} {"#feats":>6}  {"Val":>6}  {"Test":>6}')
print('  ' + '-'*44)
results = {}
for name, fcols in FEAT_SETS.items():
    va, te, *_ = run_facial_hmm(fcols, n_comp=3)
    results[name] = (va, te, len(fcols))
    print(f'  {name:<20} {len(fcols):>6}  {va:>6.4f}  {te:>6.4f}')

best_feat = max(results, key=lambda k: results[k][0])
best_fcols = FEAT_SETS[best_feat]
print(f'\nBest feature set on Val: {best_feat}  (Val={results[best_feat][0]:.4f})')

# ── sweep n_components with best feature set ─────────────────────────
print(f'\nn_components sweep (feature set = {best_feat}):')
print(f'  {"n_comp":>6}  {"Val":>6}  {"Test":>6}')
print('  ' + '-'*22)
best_n, best_n_acc = 3, 0
for n in [2, 3, 4, 5]:
    va, te, mdls, sc_best, yva_true, yva_pred = run_facial_hmm(best_fcols, n_comp=n)
    print(f'  {n:>6}  {va:>6.4f}  {te:>6.4f}')
    if va > best_n_acc:
        best_n_acc, best_n = va, n
        best_models, best_sc = mdls, sc_best
        best_yva_true, best_yva_pred = yva_true, yva_pred

print(f'\nBest config: feature={best_feat}, n_comp={best_n}')

# ── final report ─────────────────────────────────────────────────────
va_f, te_f, _, _, yva_t, yva_p = run_facial_hmm(best_fcols, n_comp=best_n)
yte_true = fte['facial_label'].values
_sc2 = StandardScaler()
Xva_f = _sc2.fit_transform(ftr[best_fcols].values)
Xva_f = _sc2.transform(fva[best_fcols].values)
Xte_f = _sc2.transform(fte[best_fcols].values)

print('\nBest HMM — Validation Classification Report:')
print(classification_report(yva_t, yva_p, zero_division=0))

# ── summary ──────────────────────────────────────────────────────────
print(f'{"Model":<30} {"Val":>6}  {"Test":>6}')
print('-' * 50)
d_va = accuracy_score(_yva, _d.predict(_Xva))
d_te = accuracy_score(_yte, _d.predict(_Xte))
print(f'{"Dummy":<30} {d_va:>6.4f}  {d_te:>6.4f}')
print(f'{"RF (merged labels)":<30} {rf_fval:>6.4f}  {rf_ftest:>6.4f}')
print(f'{"HMM best (" + best_feat + ", n=" + str(best_n) + ")":<30} {va_f:>6.4f}  {te_f:>6.4f}')


Facial classes after merge: ['big_smile', 'frown', 'neutral', 'smile']

Training distribution:
  smile       : 104  (33.1%)
  neutral     :  91  (29.0%)
  big_smile   :  63  (20.1%)
  frown       :  56  (17.8%)

Dummy — Val: 0.2031  Test: 0.1538
RF    — Val: 0.2031  Test: 0.2115

Feature set sweep (n_components=3):
  Feature set          #feats     Val    Test
  --------------------------------------------
  all_111                 111  0.2812  0.3462


Model is not converging.  Current: -3773.518616173963 is not greater than -3773.5186120411863. Delta is -4.13277666666545e-06


  mfcc_only                26  0.1875  0.1538
  mfcc+chroma              50  0.3594  0.3077
  mfcc+delta               78  0.2031  0.3269


Model is not converging.  Current: -5522.036225948872 is not greater than -5522.036225229848. Delta is -7.190237738541327e-07


  mfcc+spectral            35  0.3125  0.3269
  no_delta2                85  0.2812  0.2308

Best feature set on Val: mfcc+chroma  (Val=0.3594)

n_components sweep (feature set = mfcc+chroma):
  n_comp     Val    Test
  ----------------------


Model is not converging.  Current: -3773.518616173963 is not greater than -3773.5186120411863. Delta is -4.13277666666545e-06
Model is not converging.  Current: -3128.005451877933 is not greater than -3128.005428237958. Delta is -2.363997509746696e-05


       2  0.3906  0.3462
       3  0.3594  0.3077
       4  0.2500  0.2500
       5  0.2188  0.2500

Best config: feature=mfcc+chroma, n_comp=2

Best HMM — Validation Classification Report:
              precision    recall  f1-score   support

   big_smile       0.25      0.07      0.11        15
       frown       0.34      0.61      0.44        18
     neutral       0.46      0.67      0.55        18
       smile       0.50      0.08      0.13        13

    accuracy                           0.39        64
   macro avg       0.39      0.36      0.31        64
weighted avg       0.39      0.39      0.33        64

Model                             Val    Test
--------------------------------------------------
Dummy                          0.2031  0.1538
RF (merged labels)             0.2031  0.2115
HMM best (mfcc+chroma, n=2)    0.3906  0.3462


# Facial Expression — HMM Results

## Class Merging
Rare classes merged before training (too few samples for HMM):

| Original | Merged into | Reason |
|----------|-------------|--------|
| `negative` (6) | `frown` | both negative valence |
| `thoughtful` (22) | `neutral` | both low-arousal |
| `surprise` (30) | `big_smile` | both high-arousal positive |

Final 4 classes: `smile` (33%), `neutral` (29%), `big_smile` (20%), `frown` (18%)

## Feature Set Sweep (n_components=3)

| Feature set | #feats | Val | Test |
|-------------|--------|-----|------|
| mfcc_only | 26 | 0.19 | 0.15 |
| mfcc+delta | 78 | 0.20 | 0.33 |
| no_delta2 | 85 | 0.28 | 0.23 |
| all_111 | 111 | 0.28 | 0.35 |
| mfcc+spectral | 35 | 0.31 | 0.33 |
| **mfcc+chroma** | **50** | **0.36** | **0.31** |

## n_components Sweep (mfcc+chroma)

| n_comp | Val | Test |
|--------|-----|------|
| **2** | **0.39** | **0.35** |
| 3 | 0.36 | 0.31 |
| 4 | 0.25 | 0.25 |
| 5 | 0.22 | 0.25 |

More hidden states = worse. With ~60 samples per class, complex models overfit.

## Summary

| Model | Val | Test |
|-------|-----|------|
| Dummy | 0.20 | 0.15 |
| RF | 0.20 | 0.21 |
| **HMM (mfcc+chroma, n=2)** | **0.39** | **0.35** |

HMM significantly outperforms RF on facial expression — RF collapsed to predicting `smile` for everything; HMM with a simpler feature set generalises better.

In [4]:

# ════════════════════════════════════════════════════════════════════
# Head Movement — HMM feature set sweep
# (reuses class_hmms training setup from cell above, head labels only)
# ════════════════════════════════════════════════════════════════════

# ── head data (same merge as before) ────────────────────────────────
h = pd.read_csv(FEATURES_PATH)
hm = pd.read_csv(METADATA_PATH)
h['head_movement'] = h['head_movement'].str.strip()
h['split'] = h['song_name'].map(dict(zip(hm['title'], hm['split'])))
head_merge = {'shake, nod': 'nod', 'sway, nod': 'sway',
              'look down': 'none', 'look up': 'none'}
h['head_label'] = h['head_movement'].replace(head_merge)
HEAD_CLASSES = sorted(h['head_label'].unique())

htr = h[h['split'] == 'Train']
hva = h[h['split'] == 'Validation']
hte = h[h['split'] == 'Test']

HLABEL_COLS = ['song_name', 'start_time', 'end_time', 'duration',
               'head_movement', 'facial_expression', 'intensity',
               'head_label', 'split']
H_ALL = [c for c in h.columns if c not in HLABEL_COLS]

h_mfcc     = [c for c in H_ALL if c.startswith('mfcc') and 'delta' not in c]
h_delta    = [c for c in H_ALL if 'delta' in c]
h_chroma   = [c for c in H_ALL if c.startswith('chroma')]
h_spectral = [c for c in H_ALL if any(c.startswith(p) for p in
               ['rms', 'spectral', 'zcr', 'tempo'])]

H_FEAT_SETS = {
    'all_111':       H_ALL,
    'mfcc_only':     h_mfcc,
    'mfcc+chroma':   h_mfcc + h_chroma,
    'mfcc+delta':    h_mfcc + h_delta,
    'mfcc+spectral': h_mfcc + h_spectral,
    'chroma+spectral': h_chroma + h_spectral,
    'no_delta2':     [c for c in H_ALL if 'delta2' not in c],
}

# train/eval helper 
def run_head_hmm(feat_cols, n_comp=3):
    sc = StandardScaler()
    Xtr = sc.fit_transform(htr[feat_cols].values)
    Xva = sc.transform(hva[feat_cols].values)
    Xte = sc.transform(hte[feat_cols].values)
    ytr = htr['head_label'].values
    yva = hva['head_label'].values
    yte = hte['head_label'].values

    models = {}
    for cls in HEAD_CLASSES:
        segs = Xtr[ytr == cls]
        if len(segs) < n_comp:
            continue
        k = min(n_comp, len(segs))
        m = hmm.GaussianHMM(n_components=k, covariance_type='diag',
                             n_iter=200, random_state=42, verbose=False)
        m.fit(segs, [len(segs)])
        models[cls] = m

    def predict(X):
        preds = []
        for x in X:
            obs = x.reshape(1, -1)
            best, best_s = None, -np.inf
            for cls, m in models.items():
                try: s = m.score(obs)
                except: s = -np.inf
                if s > best_s:
                    best_s, best = s, cls
            preds.append(best)
        return np.array(preds)

    return (accuracy_score(yva, predict(Xva)),
            accuracy_score(yte, predict(Xte)),
            yva, predict(Xva))

# feature set sweep (n_comp=3) 
print('Head HMM — Feature set sweep (n_components=3)')
print(f'  {"Feature set":<20} {"#feats":>6}  {"Val":>6}  {"Test":>6}')
print('  ' + '-' * 44)
h_results = {}
for name, fcols in H_FEAT_SETS.items():
    va, te, *_ = run_head_hmm(fcols, n_comp=3)
    h_results[name] = (va, te, len(fcols))
    print(f'  {name:<20} {len(fcols):>6}  {va:>6.4f}  {te:>6.4f}')

h_best_feat = max(h_results, key=lambda k: h_results[k][0])
h_best_fcols = H_FEAT_SETS[h_best_feat]
print(f'\nBest feature set on Val: {h_best_feat}  (Val={h_results[h_best_feat][0]:.4f})')

# n_components sweep with best feature set
print(f'\nHead HMM — n_components sweep (feature set = {h_best_feat})')
print(f'  {"n_comp":>6}  {"Val":>6}  {"Test":>6}')
print('  ' + '-' * 22)
h_best_n, h_best_acc = 3, 0
h_best_yva, h_best_ypred = None, None
for n in [2, 3, 4, 5]:
    va, te, yva_t, yva_p = run_head_hmm(h_best_fcols, n_comp=n)
    print(f'  {n:>6}  {va:>6.4f}  {te:>6.4f}')
    if va > h_best_acc:
        h_best_acc, h_best_n = va, n
        h_best_yva, h_best_ypred = yva_t, yva_p

print(f'\nBest config: feature={h_best_feat}, n_comp={h_best_n}')

# ── classification report for best config ────────────────────────────
print(f'\nBest Head HMM — Validation Classification Report:')
print(classification_report(h_best_yva, h_best_ypred, zero_division=0))

# ── reference: original HMM (all_111, n=3) ───────────────────────────
orig_va, orig_te, *_ = run_head_hmm(H_ALL, n_comp=3)
print(f'{"Model":<35} {"Val":>6}  {"Test":>6}')
print(f'{"HMM original (all_111, n=3)":<35} {orig_va:>6.4f}  {orig_te:>6.4f}')
va_best, te_best, *_ = run_head_hmm(h_best_fcols, n_comp=h_best_n)
print(f'{"HMM best (" + h_best_feat + ", n=" + str(h_best_n) + ")":<35} {va_best:>6.4f}  {te_best:>6.4f}')


Model is not converging.  Current: -6492.128725261502 is not greater than -6492.128613162446. Delta is -0.00011209905642317608
Model is not converging.  Current: -929.5771622826913 is not greater than -929.5771546739562. Delta is -7.6087351317255525e-06


Head HMM — Feature set sweep (n_components=3)
  Feature set          #feats     Val    Test
  --------------------------------------------
  all_111                 111  0.2969  0.3462
  mfcc_only                26  0.2656  0.3077
  mfcc+chroma              50  0.3750  0.3269
  mfcc+delta               78  0.2344  0.3654
  mfcc+spectral            35  0.1562  0.2308
  chroma+spectral          33  0.3125  0.2885
  no_delta2                85  0.2969  0.3654

Best feature set on Val: mfcc+chroma  (Val=0.3750)

Head HMM — n_components sweep (feature set = mfcc+chroma)
  n_comp     Val    Test
  ----------------------
       2  0.2969  0.3077
       3  0.3750  0.3269


Model is not converging.  Current: -1598.3655988968471 is not greater than -1598.3655971684334. Delta is -1.7284137356909923e-06
Model is not converging.  Current: -6492.128725261502 is not greater than -6492.128613162446. Delta is -0.00011209905642317608


       4  0.2812  0.2885
       5  0.3594  0.2115

Best config: feature=mfcc+chroma, n_comp=3

Best Head HMM — Validation Classification Report:
              precision    recall  f1-score   support

         nod       0.33      0.04      0.08        23
        none       0.67      0.40      0.50        20
       shake       0.27      0.75      0.40         8
        sway       0.33      0.69      0.45        13

    accuracy                           0.38        64
   macro avg       0.40      0.47      0.36        64
weighted avg       0.43      0.38      0.33        64

Model                                  Val    Test
HMM original (all_111, n=3)         0.2969  0.3462
HMM best (mfcc+chroma, n=3)         0.3750  0.3269


# Head Movement — HMM Feature Sweep Results

## Feature Set Sweep (n_components=3)

| Feature set | #feats | Val | Test |
|-------------|--------|-----|------|
| mfcc+spectral | 35 | 0.16 | 0.23 |
| mfcc_only | 26 | 0.27 | 0.31 |
| chroma+spectral | 33 | 0.31 | 0.29 |
| all_111 | 111 | 0.30 | 0.35 |
| no_delta2 | 85 | 0.30 | 0.37 |
| **mfcc+chroma** | **50** | **0.38** | **0.33** |

## n_components Sweep (mfcc+chroma)

| n_comp | Val | Test |
|--------|-----|------|
| 2 | 0.30 | 0.31 |
| **3** | **0.38** | **0.33** |
| 4 | 0.28 | 0.29 |
| 5 | 0.36 | 0.21 |

## Per-Class Recall (best config: mfcc+chroma, n=3)

| Class | Recall | Note |
|-------|--------|------|
| shake | **0.75** | distinctive audio pattern |
| sway | **0.69** | persistent motion, HMM captures it |
| none | 0.40 | moderate |
| nod | 0.04 | near-zero — instantaneous motion, looks like `none` acoustically |

## Key Finding

`mfcc+chroma` (50 features) is the best feature set for **both facial and head** tasks.
Delta/delta2 features add noise rather than signal for per-class HMMs — removing them improves performance.

HMM is better than RF for sustained motions (`sway`, `shake`), but fails on instantaneous ones (`nod`).

In [5]:

# ════════════════════════════════════════════════════════════════════
# Experiment 1: Head RF with mfcc+chroma only (50 features)
# Experiment 2: Hybrid — Head=RF(best feat), Facial=HMM(mfcc+chroma,n=2)
# ════════════════════════════════════════════════════════════════════

# ── Exp 1: Head RF on mfcc+chroma ────────────────────────────────────
print('Experiment 1: Head RF — mfcc+chroma vs all_111')
print('─' * 50)

def run_head_rf(feat_cols, label=''):
    sc = StandardScaler()
    Xtr = sc.fit_transform(htr[feat_cols].values)
    Xva = sc.transform(hva[feat_cols].values)
    Xte = sc.transform(hte[feat_cols].values)
    ytr_ = htr['head_label'].values
    yva_ = hva['head_label'].values
    yte_ = hte['head_label'].values

    clf = RandomForestClassifier(n_estimators=200, max_depth=20,
                                  min_samples_split=5, min_samples_leaf=2,
                                  class_weight='balanced',
                                  random_state=42, n_jobs=-1)
    clf.fit(Xtr, ytr_)
    va = accuracy_score(yva_, clf.predict(Xva))
    te = accuracy_score(yte_, clf.predict(Xte))
    return va, te, clf, sc, yva_, clf.predict(Xva), yte_, clf.predict(Xte)

rf_all_va,    rf_all_te,    *_            = run_head_rf(H_ALL,           'all_111')
rf_mc_va,     rf_mc_te,     rf_mc, sc_mc, \
    hrf_yva, hrf_ypva, hrf_yte, hrf_ypte  = run_head_rf(h_mfcc+h_chroma, 'mfcc+chroma')

print(f'  RF all_111    — Val: {rf_all_va:.4f}  Test: {rf_all_te:.4f}')
print(f'  RF mfcc+chroma— Val: {rf_mc_va:.4f}  Test: {rf_mc_te:.4f}')
print()
print('RF mfcc+chroma — Validation Classification Report:')
print(classification_report(hrf_yva, hrf_ypva, zero_division=0))

# ── Exp 2: Hybrid predictor ──────────────────────────────────────────
print()
print('Experiment 2: Hybrid (Head=RF mfcc+chroma, Facial=HMM mfcc+chroma n=2)')
print('─' * 50)

# Re-train facial HMM (mfcc+chroma, n=2) — best from facial sweep
f_mc_cols = [c for c in f2.columns
             if c not in LABEL_COLS2
             and (c.startswith('mfcc') and 'delta' not in c
                  or c.startswith('chroma'))]

fac_sc = StandardScaler()
fac_Xtr = fac_sc.fit_transform(ftr[f_mc_cols].values)
fac_Xva = fac_sc.transform(fva[f_mc_cols].values)
fac_Xte = fac_sc.transform(fte[f_mc_cols].values)
fac_ytr = ftr['facial_label'].values
fac_yva = fva['facial_label'].values
fac_yte = fte['facial_label'].values

fac_models = {}
for cls in FACIAL_CLASSES:
    segs = fac_Xtr[fac_ytr == cls]
    if len(segs) < 2: continue
    m = hmm.GaussianHMM(n_components=2, covariance_type='diag',
                         n_iter=200, random_state=42, verbose=False)
    m.fit(segs, [len(segs)])
    fac_models[cls] = m

def hmm_predict_single(X, models):
    preds = []
    for x in X:
        obs = x.reshape(1, -1)
        best, best_s = None, -np.inf
        for cls, m in models.items():
            try: s = m.score(obs)
            except: s = -np.inf
            if s > best_s:
                best_s, best = s, cls
        preds.append(best)
    return np.array(preds)

fac_val_pred  = hmm_predict_single(fac_Xva, fac_models)
fac_test_pred = hmm_predict_single(fac_Xte, fac_models)

fac_val_acc  = accuracy_score(fac_yva, fac_val_pred)
fac_test_acc = accuracy_score(fac_yte, fac_test_pred)

print(f'  Head  RF  (mfcc+chroma) — Val: {rf_mc_va:.4f}  Test: {rf_mc_te:.4f}')
print(f'  Facial HMM (mfcc+chroma, n=2) — Val: {fac_val_acc:.4f}  Test: {fac_test_acc:.4f}')

print()
print('Facial HMM — Validation Classification Report:')
print(classification_report(fac_yva, fac_val_pred, zero_division=0))
print('Head RF — Validation Classification Report:')
print(classification_report(hrf_yva, hrf_ypva, zero_division=0))

# ── full summary ──────────────────────────────────────────────────────
print()
print('=' * 55)
print(f'{"Task / Model":<40} {"Val":>6}  {"Test":>6}')
print('-' * 55)
print(f'{"HEAD  Dummy":<40} {accuracy_score(hva["head_label"], ["nod"]*len(hva)):>6.4f}  —')
orig_rf_va, orig_rf_te, *_ = run_head_rf(H_ALL)
print(f'{"HEAD  RF all_111":<40} {orig_rf_va:>6.4f}  {orig_rf_te:>6.4f}')
print(f'{"HEAD  HMM all_111 n=3":<40} {orig_va:>6.4f}  {orig_te:>6.4f}')
print(f'{"HEAD  HMM mfcc+chroma n=3":<40} {h_best_acc:>6.4f}  —')
print(f'{"HEAD  RF  mfcc+chroma  [Exp 1]":<40} {rf_mc_va:>6.4f}  {rf_mc_te:>6.4f}')
print('-' * 55)
print(f'{"FACIAL RF all_111 (7 classes)":<40} {"0.2344":>6}  {"0.1731":>6}')
print(f'{"FACIAL RF merged (4 classes)":<40} {rf_fval:>6.4f}  {rf_ftest:>6.4f}')
print(f'{"FACIAL HMM mfcc+chroma n=2 [Exp 2]":<40} {fac_val_acc:>6.4f}  {fac_test_acc:>6.4f}')



Experiment 1: Head RF — mfcc+chroma vs all_111
──────────────────────────────────────────────────
  RF all_111    — Val: 0.5156  Test: 0.5385
  RF mfcc+chroma— Val: 0.4219  Test: 0.5385

RF mfcc+chroma — Validation Classification Report:
              precision    recall  f1-score   support

         nod       0.42      0.48      0.45        23
        none       0.46      0.60      0.52        20
       shake       0.33      0.38      0.35         8
        sway       0.33      0.08      0.12        13

    accuracy                           0.42        64
   macro avg       0.39      0.38      0.36        64
weighted avg       0.41      0.42      0.39        64


Experiment 2: Hybrid (Head=RF mfcc+chroma, Facial=HMM mfcc+chroma n=2)
──────────────────────────────────────────────────
  Head  RF  (mfcc+chroma) — Val: 0.4219  Test: 0.5385
  Facial HMM (mfcc+chroma, n=2) — Val: 0.3906  Test: 0.3462

Facial HMM — Validation Classification Report:
              precision    recall  f1-scor

# Experiment Results

## 1: Head RF with mfcc+chroma vs all_111

| Model | Val | Test |
|-------|-----|------|
| RF all_111 | **0.52** | **0.54** |
| RF mfcc+chroma | 0.42 | 0.54 |

Reducing features hurts RF on val — RF benefits from more features since it selects the useful ones internally via feature importance. `mfcc+chroma` only helps HMM, not RF.

## 2: Hybrid Summary

| Task | Best Model | Val | Test |
|------|-----------|-----|------|
| Head | RF all_111 | **0.52** | **0.54** |
| Facial | HMM mfcc+chroma n=2 | **0.39** | **0.35** |

## Final Recommended Configuration

| Task | Model | Features | Val | Test |
|------|-------|----------|-----|------|
| Head movement | RF | all_111 | 0.52 | 0.54 |
| Facial expression | HMM | mfcc+chroma (50) | 0.39 | 0.35 |

**Different models suit different tasks:**
- Head: RF wins — `nod`/`none` are instantaneous, RF's per-segment discrimination is better
- Facial: HMM wins — fewer features reduce noise; RF collapses to predicting `smile` for everything

Both models comfortably beat their respective dummy baselines (0.31 head, 0.20 facial).

# PCA 

In [6]:

# ════════════════════════════════════════════════════════════════════
# PCA + RF and PCA + HMM — Head & Facial
# ════════════════════════════════════════════════════════════════════
from sklearn.decomposition import PCA

# ── shared helper ─────────────────────────────────────────────────────
def fit_pca(X_train, X_val, X_test, variance=0.90):
    """Fit PCA on training set, transform all splits. Returns transformed arrays + pca object."""
    pca = PCA(n_components=variance, random_state=42)
    Xtr = pca.fit_transform(X_train)
    Xva = pca.transform(X_val)
    Xte = pca.transform(X_test)
    return Xtr, Xva, Xte, pca

def scree_info(pca, label):
    cum = np.cumsum(pca.explained_variance_ratio_)
    print(f'{label}: {pca.n_components_} components explain '
          f'{cum[-1]*100:.1f}% variance  (from 111 features)')

# ════════════════════════════════════════════════════════════════════
# HEAD — PCA + RF
# ════════════════════════════════════════════════════════════════════
print('─' * 55)
print('HEAD — PCA + RF')
print('─' * 55)

# Scale with all 111 features
h_sc_all = StandardScaler()
hXtr_s = h_sc_all.fit_transform(htr[H_ALL].values)
hXva_s = h_sc_all.transform(hva[H_ALL].values)
hXte_s = h_sc_all.transform(hte[H_ALL].values)
hytr = htr['head_label'].values
hyva = hva['head_label'].values
hyte = hte['head_label'].values

# PCA
hXtr_p, hXva_p, hXte_p, h_pca = fit_pca(hXtr_s, hXva_s, hXte_s, variance=0.90)
scree_info(h_pca, 'Head PCA')

# RF on PCA features
h_rf_pca = RandomForestClassifier(n_estimators=200, max_depth=20,
                                   min_samples_split=5, min_samples_leaf=2,
                                   class_weight='balanced',
                                   random_state=42, n_jobs=-1)
h_rf_pca.fit(hXtr_p, hytr)
h_pca_rf_train = accuracy_score(hytr, h_rf_pca.predict(hXtr_p))
h_pca_rf_val   = accuracy_score(hyva, h_rf_pca.predict(hXva_p))
h_pca_rf_test  = accuracy_score(hyte, h_rf_pca.predict(hXte_p))

print(f'\nRF + PCA — Train: {h_pca_rf_train:.4f}  Val: {h_pca_rf_val:.4f}  Test: {h_pca_rf_test:.4f}')
print(f'RF no PCA — Train: 1.0000  Val: {rf_all_va:.4f}  Test: {rf_all_te:.4f}')
print('\nHead RF+PCA — Validation Classification Report:')
print(classification_report(hyva, h_rf_pca.predict(hXva_p), zero_division=0))

# ════════════════════════════════════════════════════════════════════
# HEAD — PCA + HMM
# ════════════════════════════════════════════════════════════════════
print('HEAD — PCA + HMM')

h_hmm_pca_models = {}
for cls in HEAD_CLASSES:
    segs = hXtr_p[hytr == cls]
    if len(segs) < 3: continue
    m = hmm.GaussianHMM(n_components=3, covariance_type='diag',
                         n_iter=200, random_state=42, verbose=False)
    m.fit(segs, [len(segs)])
    h_hmm_pca_models[cls] = m

h_pca_hmm_val  = accuracy_score(hyva, hmm_predict_single(hXva_p, h_hmm_pca_models))
h_pca_hmm_test = accuracy_score(hyte, hmm_predict_single(hXte_p, h_hmm_pca_models))
print(f'HMM + PCA — Val: {h_pca_hmm_val:.4f}  Test: {h_pca_hmm_test:.4f}')
print(f'HMM no PCA (mfcc+chroma, n=3) — Val: {h_best_acc:.4f}')

print('\nHead HMM+PCA — Validation Classification Report:')
print(classification_report(hyva, hmm_predict_single(hXva_p, h_hmm_pca_models), zero_division=0))

# ════════════════════════════════════════════════════════════════════
# FACIAL — PCA + RF
# ════════════════════════════════════════════════════════════════════
print('FACIAL — PCA + RF')

f_sc_all = StandardScaler()
fXtr_s = f_sc_all.fit_transform(ftr[ALL_FEATS].values)
fXva_s = f_sc_all.transform(fva[ALL_FEATS].values)
fXte_s = f_sc_all.transform(fte[ALL_FEATS].values)
fytr = ftr['facial_label'].values
fyva = fva['facial_label'].values
fyte = fte['facial_label'].values

fXtr_p, fXva_p, fXte_p, f_pca = fit_pca(fXtr_s, fXva_s, fXte_s)
scree_info(f_pca, 'Facial PCA')


f_rf_pca = RandomForestClassifier(n_estimators=200, max_depth=10,
                                   min_samples_split=5, min_samples_leaf=2,
                                   max_features='sqrt',
                                   random_state=42, n_jobs=-1)
f_rf_pca.fit(fXtr_p, fytr)
f_pca_rf_train = accuracy_score(fytr, f_rf_pca.predict(fXtr_p))
f_pca_rf_val   = accuracy_score(fyva, f_rf_pca.predict(fXva_p))
f_pca_rf_test  = accuracy_score(fyte, f_rf_pca.predict(fXte_p))

print(f'\nRF + PCA — Train: {f_pca_rf_train:.4f}  Val: {f_pca_rf_val:.4f}  Test: {f_pca_rf_test:.4f}')
print(f'RF no PCA — Train: 0.9936  Val: {rf_fval:.4f}  Test: {rf_ftest:.4f}')
print('\nFacial RF+PCA — Validation Classification Report:')
print(classification_report(fyva, f_rf_pca.predict(fXva_p), zero_division=0))

# FACIAL — PCA + HMM
print('FACIAL — PCA + HMM')

f_hmm_pca_models = {}
for cls in FACIAL_CLASSES:
    segs = fXtr_p[fytr == cls]
    if len(segs) < 2: continue
    m = hmm.GaussianHMM(n_components=2, covariance_type='diag',
                         n_iter=200, random_state=42, verbose=False)
    m.fit(segs, [len(segs)])
    f_hmm_pca_models[cls] = m

f_pca_hmm_val  = accuracy_score(fyva, hmm_predict_single(fXva_p, f_hmm_pca_models))
f_pca_hmm_test = accuracy_score(fyte, hmm_predict_single(fXte_p, f_hmm_pca_models))
print(f'HMM + PCA — Val: {f_pca_hmm_val:.4f}  Test: {f_pca_hmm_test:.4f}')
print(f'HMM no PCA (mfcc+chroma, n=2) — Val: {fac_val_acc:.4f}  Test: {fac_test_acc:.4f}')

print('\nFacial HMM+PCA — Validation Classification Report:')
print(classification_report(fyva, hmm_predict_single(fXva_p, f_hmm_pca_models), zero_division=0))

# Full summary
print()
print(f'{"Task / Model":<42} {"Val":>7}  {"Test":>7}')
print('-' * 60)
print(f'{"HEAD  RF  all_111 (no PCA)":<42} {rf_all_va:>7.4f}  {rf_all_te:>7.4f}')
print(f'{"HEAD  RF  + PCA (90%)":<42} {h_pca_rf_val:>7.4f}  {h_pca_rf_test:>7.4f}')
print(f'{"HEAD  HMM mfcc+chroma n=3 (no PCA)":<42} {h_best_acc:>7.4f}  —')
print(f'{"HEAD  HMM + PCA (90%)":<42} {h_pca_hmm_val:>7.4f}  {h_pca_hmm_test:>7.4f}')
print('-' * 60)
print(f'{"FACIAL RF  merged (no PCA)":<42} {rf_fval:>7.4f}  {rf_ftest:>7.4f}')
print(f'{"FACIAL RF  + PCA (90%)":<42} {f_pca_rf_val:>7.4f}  {f_pca_rf_test:>7.4f}')
print(f'{"FACIAL HMM mfcc+chroma n=2 (no PCA)":<42} {fac_val_acc:>7.4f}  {fac_test_acc:>7.4f}')
print(f'{"FACIAL HMM + PCA (90%)":<42} {f_pca_hmm_val:>7.4f}  {f_pca_hmm_test:>7.4f}')


───────────────────────────────────────────────────────
HEAD — PCA + RF
───────────────────────────────────────────────────────
Head PCA: 38 components explain 90.4% variance  (from 111 features)

RF + PCA — Train: 1.0000  Val: 0.4688  Test: 0.4423
RF no PCA — Train: 1.0000  Val: 0.5156  Test: 0.5385

Head RF+PCA — Validation Classification Report:
              precision    recall  f1-score   support

         nod       0.44      0.52      0.48        23
        none       0.50      0.70      0.58        20
       shake       0.20      0.12      0.15         8
        sway       0.75      0.23      0.35        13

    accuracy                           0.47        64
   macro avg       0.47      0.39      0.39        64
weighted avg       0.49      0.47      0.45        64

HEAD — PCA + HMM
HMM + PCA — Val: 0.4375  Test: 0.3654
HMM no PCA (mfcc+chroma, n=3) — Val: 0.3750

Head HMM+PCA — Validation Classification Report:
              precision    recall  f1-score   support

         n


## Summary

### Best Models Per Task

| Task | Model | Features | Val Acc | Test Acc |
|------|-------|----------|---------|----------|
| Head Movement | Random Forest | all_111 | 51.6% | 53.8% |
| Facial Expression | HMM (n=2) | mfcc+chroma | 39.1% | 34.6% |

*(Baseline: 25% for random guessing)*


### Key Findings

**1. Different tasks need different models:**
- **Head movements** are instantaneous, discrete actions → RF works better
- **Facial expressions** have weak audio correlation → HMM with fewer features reduces noise

**2. Feature selection matters:**
- **RF**: Benefits from all 111 features (has built-in feature selection)
- **HMM**: Works best with 50 features (mfcc+chroma) - too many features add noise

**3. PCA dimensionality reduction failed:**
- Head RF: 51.6% → 43.8%
- Facial HMM: 39.1% → 31.2% (worse by 40%)
- Reason: Small dataset + PCA discards discriminative features with low variance

**4. Model parameters:**
- Head: n_components = 3 states per class
- Facial: n_components = 2 states per class
- Fewer samples → fewer states needed to avoid overfitting

---

### Performance by Class

**Head Movement (RF):**
- Best: `none` (60% recall), `nod` (48% recall)
- Worst: `shake` (38% recall), `sway` (8% recall)

**Facial Expression (HMM):**
- Best: `neutral` (40% recall)
- Worst: All classes struggle - weak audio-expression correlation


# new comparision

# Facial Expression — 7-class vs 4-class Merge + SVM Comparison

Three merging strategies are compared:
- **7-class**: negative, big_smile, surprise, frown, thoughtful, smile, neutral
- **4-class**: smile+big_smile → `smile_pos`, neutral+thoughtful → `neutral_calm`, frown, surprise
- Models: HMM (mfcc+chroma, best config), RF, SVM

In [7]:
# ════════════════════════════════════════════════════════════════════
# Facial Expression — 7-class / 4-class merge comparison
# Models: HMM (mfcc+chroma, n=2), RF, SVM
# ════════════════════════════════════════════════════════════════════
from sklearn.svm import SVC

# ── reload raw data ──────────────────────────────────────────────────
fa = pd.read_csv(FEATURES_PATH)
ma = pd.read_csv(METADATA_PATH)
fa['facial_expression'] = fa['facial_expression'].str.strip()
fa['split'] = fa['song_name'].map(dict(zip(ma['title'], ma['split'])))

# ── Merge A: 7-class (as specified) ─────────────────────────────────
def merge_facial_7(expression):
    expression = str(expression).strip().lower()
    if 'angry' in expression or 'disgust' in expression:
        return 'negative'
    elif 'big smile' in expression or ('browraise' in expression and 'smile' in expression):
        return 'big_smile'
    elif 'browraise' in expression or 'surprise' in expression or 'oh' in expression:
        return 'surprise'
    elif 'frown' in expression or 'sad' in expression:
        return 'frown'
    elif 'thoughtful' in expression or 'confused' in expression:
        return 'thoughtful'
    elif 'smile' in expression:
        return 'smile'
    elif 'neutral' in expression:
        return 'neutral'
    else:
        return 'neutral'

# ── Merge B: 4-class (smile+big_smile / neutral+thoughtful / frown / surprise) ──
def merge_facial_4(expression):
    expression = str(expression).strip().lower()
    if 'angry' in expression or 'disgust' in expression or 'frown' in expression or 'sad' in expression:
        return 'frown'
    elif 'big smile' in expression or ('browraise' in expression and 'smile' in expression) or 'smile' in expression:
        return 'smile'
    elif 'browraise' in expression or 'surprise' in expression or 'oh' in expression:
        return 'surprise'
    else:
        return 'neutral'

fa['label_7'] = fa['facial_expression'].apply(merge_facial_7)
fa['label_4'] = fa['facial_expression'].apply(merge_facial_4)

# ── feature cols (mfcc+chroma, best from sweep) ──────────────────────
_exclude = ['song_name', 'start_time', 'end_time', 'duration',
            'head_movement', 'facial_expression', 'intensity',
            'label_7', 'label_4', 'split']
_all_cols = [c for c in fa.columns if c not in _exclude]
mc_cols = [c for c in _all_cols if (c.startswith('mfcc') and 'delta' not in c) or c.startswith('chroma')]
all_cols = _all_cols

print(f'mfcc+chroma: {len(mc_cols)} features | all: {len(all_cols)} features')

# splits
fa_tr = fa[fa['split'] == 'Train']
fa_va = fa[fa['split'] == 'Validation']
fa_te = fa[fa['split'] == 'Test']

# ── distribution ─────────────────────────────────────────────────────
for merge_name, col in [('7-class', 'label_7'), ('4-class', 'label_4')]:
    print(f'\n{merge_name} training distribution:')
    vc = fa_tr[col].value_counts()
    for lbl, cnt in vc.items():
        print(f'  {lbl:15s}: {cnt:3d}  ({cnt/len(fa_tr)*100:.1f}%)')

mfcc+chroma: 50 features | all: 111 features

7-class training distribution:
  smile          : 104  (33.1%)
  neutral        :  69  (22.0%)
  frown          :  50  (15.9%)
  big_smile      :  33  (10.5%)
  surprise       :  30  (9.6%)
  thoughtful     :  22  (7.0%)
  negative       :   6  (1.9%)

4-class training distribution:
  smile          : 136  (43.3%)
  neutral        :  91  (29.0%)
  frown          :  57  (18.2%)
  surprise       :  30  (9.6%)


In [8]:
# ════════════════════════════════════════════════════════════════════
# Helpers: train HMM / RF / SVM for a given label column + feature set
# ════════════════════════════════════════════════════════════════════

def train_eval_hmm(label_col, feat_cols, n_comp=2):
    """Train one GaussianHMM per class, score each sample independently."""
    sc = StandardScaler()
    Xtr = sc.fit_transform(fa_tr[feat_cols].values)
    Xva = sc.transform(fa_va[feat_cols].values)
    Xte = sc.transform(fa_te[feat_cols].values)
    ytr = fa_tr[label_col].values
    yva = fa_va[label_col].values
    yte = fa_te[label_col].values

    classes = sorted(fa[label_col].unique())
    models = {}
    for cls in classes:
        segs = Xtr[ytr == cls]
        if len(segs) < n_comp:
            continue
        k = min(n_comp, len(segs))
        m = hmm.GaussianHMM(n_components=k, covariance_type='diag',
                             n_iter=200, random_state=42, verbose=False)
        m.fit(segs, [len(segs)])
        models[cls] = m

    def predict(X):
        preds = []
        for x in X:
            obs = x.reshape(1, -1)
            best, best_s = None, -np.inf
            for cls, m in models.items():
                try:
                    s = m.score(obs)
                except Exception:
                    s = -np.inf
                if s > best_s:
                    best_s, best = s, cls
            preds.append(best)
        return np.array(preds)

    va = accuracy_score(yva, predict(Xva))
    te = accuracy_score(yte, predict(Xte))
    return va, te, yva, predict(Xva), yte, predict(Xte)


def train_eval_rf(label_col, feat_cols):
    sc = StandardScaler()
    Xtr = sc.fit_transform(fa_tr[feat_cols].values)
    Xva = sc.transform(fa_va[feat_cols].values)
    Xte = sc.transform(fa_te[feat_cols].values)
    ytr = fa_tr[label_col].values
    yva = fa_va[label_col].values
    yte = fa_te[label_col].values

    clf = RandomForestClassifier(n_estimators=200, max_depth=20,
                                  min_samples_split=5, min_samples_leaf=2,
                                  class_weight='balanced',
                                  random_state=42, n_jobs=-1)
    clf.fit(Xtr, ytr)
    va = accuracy_score(yva, clf.predict(Xva))
    te = accuracy_score(yte, clf.predict(Xte))
    return va, te, yva, clf.predict(Xva), yte, clf.predict(Xte)


def train_eval_svm(label_col, feat_cols):
    sc = StandardScaler()
    Xtr = sc.fit_transform(fa_tr[feat_cols].values)
    Xva = sc.transform(fa_va[feat_cols].values)
    Xte = sc.transform(fa_te[feat_cols].values)
    ytr = fa_tr[label_col].values
    yva = fa_va[label_col].values
    yte = fa_te[label_col].values

    clf = SVC(kernel='rbf', C=1.0, gamma='scale',
              class_weight='balanced', random_state=42)
    clf.fit(Xtr, ytr)
    va = accuracy_score(yva, clf.predict(Xva))
    te = accuracy_score(yte, clf.predict(Xte))
    return va, te, yva, clf.predict(Xva), yte, clf.predict(Xte)


print('Helpers defined.')

# dummy baseline helper
def dummy_acc(label_col):
    ytr = fa_tr[label_col].values
    yva = fa_va[label_col].values
    yte = fa_te[label_col].values
    d = DummyClassifier(strategy='most_frequent', random_state=42)
    d.fit(fa_tr[mc_cols].values, ytr)
    return accuracy_score(yva, d.predict(fa_va[mc_cols].values)), \
           accuracy_score(yte, d.predict(fa_te[mc_cols].values))

Helpers defined.


In [11]:
# ════════════════════════════════════════════════════════════════════
# Run all experiments: 7-class vs 4-class × HMM / RF / SVM
# ════════════════════════════════════════════════════════════════════
from IPython.display import display
import pandas as pd

results = []

for merge_name, label_col in [('7-class', 'label_7'), ('4-class', 'label_4')]:
    print(f'\n{"═"*60}')
    print(f'  {merge_name} — {sorted(fa[label_col].unique())}')
    print(f'{"═"*60}')

    dv, dt = dummy_acc(label_col)
    print(f'  Dummy      — Val: {dv:.4f}  Test: {dt:.4f}')
    results.append({'merge': merge_name, 'model': 'Dummy', 'val': dv, 'test': dt})

    n_comp = 2
    hv, ht, hva_y, hva_p, hte_y, hte_p = train_eval_hmm(label_col, mc_cols, n_comp=n_comp)
    print(f'  HMM (n={n_comp})   — Val: {hv:.4f}  Test: {ht:.4f}')
    results.append({'merge': merge_name, 'model': f'HMM(n={n_comp})', 'val': hv, 'test': ht})

    rv, rt, rva_y, rva_p, rte_y, rte_p = train_eval_rf(label_col, mc_cols)
    print(f'  RF (mc)    — Val: {rv:.4f}  Test: {rt:.4f}')
    results.append({'merge': merge_name, 'model': 'RF(mfcc+chroma)', 'val': rv, 'test': rt})

    rv2, rt2, rva_y2, rva_p2, rte_y2, rte_p2 = train_eval_rf(label_col, all_cols)
    print(f'  RF (all)   — Val: {rv2:.4f}  Test: {rt2:.4f}')
    results.append({'merge': merge_name, 'model': 'RF(all_111)', 'val': rv2, 'test': rt2})

    sv, st, sva_y, sva_p, ste_y, ste_p = train_eval_svm(label_col, mc_cols)
    print(f'  SVM (mc)   — Val: {sv:.4f}  Test: {st:.4f}')
    results.append({'merge': merge_name, 'model': 'SVM(mfcc+chroma)', 'val': sv, 'test': st})

    sv2, st2, sva_y2, sva_p2, ste_y2, ste_p2 = train_eval_svm(label_col, all_cols)
    print(f'  SVM (all)  — Val: {sv2:.4f}  Test: {st2:.4f}')
    results.append({'merge': merge_name, 'model': 'SVM(all_111)', 'val': sv2, 'test': st2})

    # ── 找 best model by val ──────────────────────────────────────────
    candidates = [
        ('HMM',     hv,  hva_y,  hva_p,  hte_y,  hte_p),
        ('RF(mc)',  rv,  rva_y,  rva_p,  rte_y,  rte_p),
        ('RF(all)', rv2, rva_y2, rva_p2, rte_y2, rte_p2),
        ('SVM(mc)', sv,  sva_y,  sva_p,  ste_y,  ste_p),
        ('SVM(all)',sv2, sva_y2, sva_p2, ste_y2, ste_p2),
    ]
    best_name, best_val, best_yva, best_yp_va, best_yte, best_yp_te = max(candidates, key=lambda x: x[1])

    print(f'\n  ── Best model on Val: {best_name} ({best_val:.4f}) ──')

    print(f'\n  Validation Classification Report:')
    print(classification_report(best_yva, best_yp_va, zero_division=0))

    print(f'  Test Classification Report:')
    print(classification_report(best_yte, best_yp_te, zero_division=0))

    # ── 所有模型的 val + test report 完整版 ──────────────────────────
    print(f'\n  ── Full Reports for All Models ──')
    all_models = [
        ('HMM',     hva_y,  hva_p,  hte_y,  hte_p),
        ('RF(mc)',  rva_y,  rva_p,  rte_y,  rte_p),
        ('RF(all)', rva_y2, rva_p2, rte_y2, rte_p2),
        ('SVM(mc)', sva_y,  sva_p,  ste_y,  ste_p),
        ('SVM(all)',sva_y2, sva_p2, ste_y2, ste_p2),
    ]
    for mname, va_y, va_p, te_y, te_p in all_models:
        print(f'\n  [{merge_name}] {mname} — Validation:')
        print(classification_report(va_y, va_p, zero_division=0))
        print(f'  [{merge_name}] {mname} — Test:')
        print(classification_report(te_y, te_p, zero_division=0))

# ── Summary table ─────────────────────────────────────────────────────
print('\n' + '═'*60)
print('SUMMARY TABLE')
print('═'*60)
df_res = pd.DataFrame(results)

for merge_name in ['7-class', '4-class']:
    print(f'\n  {merge_name}:')
    sub = df_res[df_res['merge'] == merge_name][['model', 'val', 'test']]
    for _, row in sub.iterrows():
        print(f'  {row["model"]:20s}  Val: {row["val"]:.4f}  Test: {row["test"]:.4f}')

Model is not converging.  Current: -1725.489259617701 is not greater than -1725.4892531139726. Delta is -6.503728400275577e-06



════════════════════════════════════════════════════════════
  7-class — ['big_smile', 'frown', 'negative', 'neutral', 'smile', 'surprise', 'thoughtful']
════════════════════════════════════════════════════════════
  Dummy      — Val: 0.2031  Test: 0.1538
  HMM (n=2)   — Val: 0.3281  Test: 0.3654
  RF (mc)    — Val: 0.1250  Test: 0.2308
  RF (all)   — Val: 0.2031  Test: 0.2500
  SVM (mc)   — Val: 0.2500  Test: 0.3269
  SVM (all)  — Val: 0.2656  Test: 0.3077

  ── Best model on Val: HMM (0.3281) ──

  Validation Classification Report:
              precision    recall  f1-score   support

   big_smile       0.31      0.89      0.46         9
       frown       0.50      0.38      0.43        16
    negative       0.00      0.00      0.00         2
     neutral       0.24      0.27      0.25        15
       smile       0.50      0.15      0.24        13
    surprise       0.20      0.17      0.18         6
  thoughtful       0.00      0.00      0.00         3

    accuracy             

# Head Movement — SVM Comparison

Add SVM to the head movement comparison (RF was previously best at Val 0.52 / Test 0.54).

In [10]:
# ════════════════════════════════════════════════════════════════════
# Head Movement — add SVM (mfcc+chroma and all_111)
# Compare with existing RF and HMM results
# ════════════════════════════════════════════════════════════════════

# ── reload head data ─────────────────────────────────────────────────
hd = pd.read_csv(FEATURES_PATH)
hmd = pd.read_csv(METADATA_PATH)
hd['head_movement'] = hd['head_movement'].str.strip()
hd['split'] = hd['song_name'].map(dict(zip(hmd['title'], hmd['split'])))
head_merge_map = {'shake, nod': 'nod', 'sway, nod': 'sway',
                  'look down': 'none', 'look up': 'none'}
hd['head_label'] = hd['head_movement'].replace(head_merge_map)
HD_CLASSES = sorted(hd['head_label'].unique())

hd_tr = hd[hd['split'] == 'Train']
hd_va = hd[hd['split'] == 'Validation']
hd_te = hd[hd['split'] == 'Test']

HD_EXCL = ['song_name', 'start_time', 'end_time', 'duration',
           'head_movement', 'facial_expression', 'intensity',
           'head_label', 'split']
HD_ALL = [c for c in hd.columns if c not in HD_EXCL]
HD_MC  = [c for c in HD_ALL if (c.startswith('mfcc') and 'delta' not in c)
          or c.startswith('chroma')]

print(f'Head classes: {HD_CLASSES}')
print(f'Features — all: {len(HD_ALL)} | mfcc+chroma: {len(HD_MC)}')

# ── SVM helper ───────────────────────────────────────────────────────
def run_head_svm(feat_cols, C=1.0, kernel='rbf', label=''):
    sc = StandardScaler()
    Xtr = sc.fit_transform(hd_tr[feat_cols].values)
    Xva = sc.transform(hd_va[feat_cols].values)
    Xte = sc.transform(hd_te[feat_cols].values)
    ytr_ = hd_tr['head_label'].values
    yva_ = hd_va['head_label'].values
    yte_ = hd_te['head_label'].values

    clf = SVC(kernel=kernel, C=C, gamma='scale',
              class_weight='balanced', random_state=42)
    clf.fit(Xtr, ytr_)
    va = accuracy_score(yva_, clf.predict(Xva))
    te = accuracy_score(yte_, clf.predict(Xte))
    return va, te, yva_, clf.predict(Xva)

# ── HMM helper (reuse pattern) ───────────────────────────────────────
def run_head_hmm_new(feat_cols, n_comp=3):
    sc = StandardScaler()
    Xtr = sc.fit_transform(hd_tr[feat_cols].values)
    Xva = sc.transform(hd_va[feat_cols].values)
    Xte = sc.transform(hd_te[feat_cols].values)
    ytr_ = hd_tr['head_label'].values
    yva_ = hd_va['head_label'].values
    yte_ = hd_te['head_label'].values

    models = {}
    for cls in HD_CLASSES:
        segs = Xtr[ytr_ == cls]
        if len(segs) < n_comp: continue
        k = min(n_comp, len(segs))
        m = hmm.GaussianHMM(n_components=k, covariance_type='diag',
                             n_iter=200, random_state=42, verbose=False)
        m.fit(segs, [len(segs)])
        models[cls] = m

    def predict(X):
        preds = []
        for x in X:
            obs = x.reshape(1, -1)
            best, best_s = None, -np.inf
            for cls, m in models.items():
                try: s = m.score(obs)
                except: s = -np.inf
                if s > best_s: best_s, best = s, cls
            preds.append(best)
        return np.array(preds)

    return accuracy_score(yva_, predict(Xva)), accuracy_score(yte_, predict(Xte)), \
           yva_, predict(Xva)

# ── RF helper ────────────────────────────────────────────────────────
def run_head_rf_new(feat_cols):
    sc = StandardScaler()
    Xtr = sc.fit_transform(hd_tr[feat_cols].values)
    Xva = sc.transform(hd_va[feat_cols].values)
    Xte = sc.transform(hd_te[feat_cols].values)
    ytr_ = hd_tr['head_label'].values
    yva_ = hd_va['head_label'].values
    yte_ = hd_te['head_label'].values

    clf = RandomForestClassifier(n_estimators=200, max_depth=20,
                                  min_samples_split=5, min_samples_leaf=2,
                                  class_weight='balanced',
                                  random_state=42, n_jobs=-1)
    clf.fit(Xtr, ytr_)
    va = accuracy_score(yva_, clf.predict(Xva))
    te = accuracy_score(yte_, clf.predict(Xte))
    return va, te, yva_, clf.predict(Xva)

# ── Dummy ─────────────────────────────────────────────────────────────
hd_dummy = DummyClassifier(strategy='most_frequent', random_state=42)
hd_dummy.fit(hd_tr[HD_ALL].values, hd_tr['head_label'].values)
dummy_hd_va = accuracy_score(hd_va['head_label'].values, hd_dummy.predict(hd_va[HD_ALL].values))
dummy_hd_te = accuracy_score(hd_te['head_label'].values, hd_dummy.predict(hd_te[HD_ALL].values))

# ── Run all ───────────────────────────────────────────────────────────
print('\n' + '═'*58)
print('HEAD MOVEMENT — Model Comparison')
print('═'*58)
print(f'  {"Model":<22} {"Features":<15} {"Val":>6}  {"Test":>6}')
print('  ' + '-'*54)

hd_results = []

print(f'  {"Dummy":<22} {"—":<15} {dummy_hd_va:.4f}  {dummy_hd_te:.4f}')
hd_results.append(('Dummy', '—', dummy_hd_va, dummy_hd_te, None, None, None))

for feat_name, feat_cols in [('mfcc+chroma', HD_MC), ('all_111', HD_ALL)]:
    # HMM
    hv, ht, hyva, hyp = run_head_hmm_new(feat_cols, n_comp=3)
    print(f'  {"HMM(n=3)":<22} {feat_name:<15} {hv:.4f}  {ht:.4f}')
    hd_results.append(('HMM(n=3)', feat_name, hv, ht, hyva, hyp, 'hmm_'+feat_name))
    # RF
    rv, rt, ryva, ryp = run_head_rf_new(feat_cols)
    print(f'  {"RF":<22} {feat_name:<15} {rv:.4f}  {rt:.4f}')
    hd_results.append(('RF', feat_name, rv, rt, ryva, ryp, 'rf_'+feat_name))
    # SVM
    sv, st, syva, syp = run_head_svm(feat_cols)
    print(f'  {"SVM(rbf)":<22} {feat_name:<15} {sv:.4f}  {st:.4f}')
    hd_results.append(('SVM(rbf)', feat_name, sv, st, syva, syp, 'svm_'+feat_name))

# SVM C sweep on all_111
print('\n  SVM C-sweep (all_111):')
for C in [0.1, 0.5, 1.0, 5.0, 10.0]:
    sv_c, st_c, *_ = run_head_svm(HD_ALL, C=C)
    print(f'    C={C:<5} Val: {sv_c:.4f}  Test: {st_c:.4f}')

# ── Best model per-class report ───────────────────────────────────────
valid_results = [(m, f, v, t, yv, yp) for m, f, v, t, yv, yp, _ in hd_results if yv is not None]
best_model = max(valid_results, key=lambda x: x[2])
bm, bf, bv, bt, byva, byp = best_model
print(f'\nBest: {bm} ({bf})  Val={bv:.4f}  Test={bt:.4f}')
print('\nValidation Classification Report:')
print(classification_report(byva, byp, zero_division=0))

Head classes: ['nod', 'none', 'shake', 'sway']
Features — all: 111 | mfcc+chroma: 50

══════════════════════════════════════════════════════════
HEAD MOVEMENT — Model Comparison
══════════════════════════════════════════════════════════
  Model                  Features           Val    Test
  ------------------------------------------------------
  Dummy                  —               0.3125  0.3077
  HMM(n=3)               mfcc+chroma     0.3750  0.3269


Model is not converging.  Current: -6492.128725261502 is not greater than -6492.128613162446. Delta is -0.00011209905642317608


  RF                     mfcc+chroma     0.4219  0.5385
  SVM(rbf)               mfcc+chroma     0.4531  0.5385
  HMM(n=3)               all_111         0.2969  0.3462
  RF                     all_111         0.5156  0.5385
  SVM(rbf)               all_111         0.5000  0.4615

  SVM C-sweep (all_111):
    C=0.1   Val: 0.4531  Test: 0.5000
    C=0.5   Val: 0.4375  Test: 0.3846
    C=1.0   Val: 0.5000  Test: 0.4615
    C=5.0   Val: 0.4531  Test: 0.5385
    C=10.0  Val: 0.4531  Test: 0.5769

Best: RF (all_111)  Val=0.5156  Test=0.5385

Validation Classification Report:
              precision    recall  f1-score   support

         nod       0.52      0.61      0.56        23
        none       0.64      0.80      0.71        20
       shake       0.12      0.12      0.12         8
        sway       0.50      0.15      0.24        13

    accuracy                           0.52        64
   macro avg       0.45      0.42      0.41        64
weighted avg       0.50      0.52      0.49 